# 🚗 Car Accident Risk Predictor

This notebook contains the Streamlit dashboard code for predicting car accident injury severity using Machine Learning.

**To run the Streamlit app, execute this in terminal:**
```bash
streamlit run app.py
```

## 1. Import Libraries

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn import tree

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

print("Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load the Monroe County Car Crash dataset
df = pd.read_csv("../monroe county car crach 2003-2015.csv", encoding='latin-1')

print(f"Dataset Shape: {df.shape}")
print(f"Total Records: {len(df):,}")
df.head()

## 3. Data Preprocessing

In [ ]:
data = df.copy()

# Map Injury Type to numeric
injury_map = {
    'No injury/unknown': 0,
    'Non-incapacitating': 1,
    'Incapacitating': 2,
    'Fatal': 3
}
data['Injury_Numeric'] = data['Injury Type'].map(injury_map).fillna(0).astype(int)

# Map Collision Type
collision_map = {
    '1-Car': 1, '2-Car': 2, '3+ Cars': 3,
    'Pedestrian': 4, 'Cyclist': 5, 'Moped/Motorcycle': 6, 'Bus': 7
}
data['Collision_Numeric'] = data['Collision Type'].map(collision_map).fillna(2).astype(int)

# Weekend binary
data['Is_Weekend'] = data['Weekend?'].apply(lambda x: 1 if x == 'Weekend' else 0)

# Extract hour
def extract_hour(h):
    try:
        h = int(h)
        return h // 100 if h >= 100 else h
    except:
        return 12
data['Hour_Numeric'] = data['Hour'].apply(extract_hour)

# Time period
def get_time_period(h):
    if 5 <= h < 12: return 0  # Morning
    elif 12 <= h < 17: return 1  # Afternoon
    elif 17 <= h < 21: return 2  # Evening
    else: return 3  # Night
data['Time_Period'] = data['Hour_Numeric'].apply(get_time_period)

# Risk factor from Primary Factor
def categorize_risk(factor):
    factor = str(factor).upper()
    high_risk = ['SPEED', 'LEFT OF CENTER', 'DISREGARD', 'RAN OFF', 'ALCOHOL', 'DRUG']
    for r in high_risk:
        if r in factor:
            return 2  # High
    medium_risk = ['YIELD', 'FOLLOWING', 'IMPROPER', 'LANE']
    for r in medium_risk:
        if r in factor:
            return 1  # Medium
    return 0  # Low

data['Risk_Factor'] = data['Primary Factor'].apply(categorize_risk)

# Rush hour
data['Is_Rush_Hour'] = data['Hour_Numeric'].apply(lambda x: 1 if (7<=x<=9) or (16<=x<=18) else 0)

print("Data preprocessing completed!")
data[['Collision_Numeric', 'Is_Weekend', 'Time_Period', 'Hour_Numeric', 'Risk_Factor', 'Is_Rush_Hour', 'Injury_Numeric']].head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Injury Type Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
data['Injury Type'].value_counts().plot(
    kind='pie',
    autopct='%1.1f%%',
    colors=['lightgreen', 'yellow', 'orange', 'red'],
    ax=axes[0]
)
axes[0].set_ylabel("")
axes[0].set_title("Injury Type Distribution", fontsize=14, fontweight='bold')

# Bar chart
data['Injury Type'].value_counts().plot(
    kind='bar',
    color=['lightgreen', 'yellow', 'orange', 'red'],
    ax=axes[1]
)
axes[1].set_title("Injury Type Counts", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Injury Type")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# Feature Correlation Heatmap
features = ['Collision_Numeric', 'Is_Weekend', 'Time_Period', 'Hour_Numeric', 'Risk_Factor', 'Is_Rush_Hour']

fig, ax = plt.subplots(figsize=(10, 6))
corr_data = data[features + ['Injury_Numeric']].corr()
sns.heatmap(corr_data, annot=True, cmap='coolwarm', ax=ax, fmt='.2f')
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Train-Test Split & Scaling

In [ ]:
# Prepare features and target
X = data[features]
y = data['Injury_Numeric']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Training set size: {len(X_train):,}")
print(f"Test set size: {len(X_test):,}")

## 6. Random Forest Model

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=50, random_state=42)
rf.fit(X_train_sc, y_train)
y_pred_rf = rf.predict(X_test_sc)

# Evaluation
print("=" * 50)
print("RANDOM FOREST MODEL EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf, average='weighted'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf, average='weighted'):.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Injury', 'Minor', 'Serious', 'Fatal'],
            yticklabels=['No Injury', 'Minor', 'Serious', 'Fatal'],
            ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Random Forest - Confusion Matrix", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Feature Importance
feat_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x='Importance', y='Feature', data=feat_df, ax=ax, palette='Reds_r')
ax.set_title("Feature Importance (Random Forest)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(feat_df)

## 7. Decision Tree Model

In [ ]:
# Train Decision Tree
dt = DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=42)
dt.fit(X_train_sc, y_train)
y_pred_dt = dt.predict(X_test_sc)

# Evaluation
print("=" * 50)
print("DECISION TREE MODEL EVALUATION")
print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt, average='weighted'):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt, average='weighted'):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt, average='weighted'):.4f}")

In [ ]:
# Decision Tree Visualization
fig, ax = plt.subplots(figsize=(20, 10))
tree.plot_tree(
    dt,
    feature_names=features,
    class_names=['No Injury', 'Minor', 'Serious', 'Fatal'],
    filled=True, rounded=True, fontsize=10, ax=ax
)
ax.set_title("Decision Tree Visualization", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. SMOTE vs NearMiss Comparison

In [ ]:
# SMOTE (Oversampling)
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X_train_sc, y_train)
rf_smote = RandomForestClassifier(n_estimators=50, random_state=42)
rf_smote.fit(X_smote, y_smote)
y_pred_smote = rf_smote.predict(X_test_sc)
acc_smote = accuracy_score(y_test, y_pred_smote)

print(f"SMOTE - Training samples: {len(X_smote):,}")
print(f"SMOTE - Accuracy: {acc_smote:.4f}")

In [ ]:
# NearMiss (Undersampling)
nm = NearMiss()
X_nm, y_nm = nm.fit_resample(X_train_sc, y_train)
rf_nm = RandomForestClassifier(n_estimators=50, random_state=42)
rf_nm.fit(X_nm, y_nm)
y_pred_nm = rf_nm.predict(X_test_sc)
acc_nm = accuracy_score(y_test, y_pred_nm)

print(f"NearMiss - Training samples: {len(X_nm):,}")
print(f"NearMiss - Accuracy: {acc_nm:.4f}")

In [ ]:
# Comparison Chart
acc_df = pd.DataFrame({
    'Method': ['Original', 'SMOTE', 'NearMiss'],
    'Accuracy': [accuracy_score(y_test, y_pred_rf), acc_smote, acc_nm]
})

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(acc_df['Method'], acc_df['Accuracy'], color=colors)
ax.set_ylim(0, 1)
ax.set_ylabel('Accuracy')
ax.set_title('Sampling Methods Comparison', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar, acc in zip(bars, acc_df['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{acc:.2%}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("CONCLUSION")
print("="*50)
if acc_smote > acc_nm:
    print(f"✅ SMOTE is recommended with higher accuracy ({acc_smote:.2%}) vs NearMiss ({acc_nm:.2%})")
else:
    print(f"✅ NearMiss is recommended with higher accuracy ({acc_nm:.2%}) vs SMOTE ({acc_smote:.2%})")

## 9. Make a Prediction

In [ ]:
# Example prediction
def predict_risk(collision, weekend, time_period, hour, risk_factor, rush_hour):
    input_data = pd.DataFrame({
        'Collision_Numeric': [collision],
        'Is_Weekend': [weekend],
        'Time_Period': [time_period],
        'Hour_Numeric': [hour],
        'Risk_Factor': [risk_factor],
        'Is_Rush_Hour': [rush_hour]
    })
    input_sc = scaler.transform(input_data)
    pred = rf.predict(input_sc)[0]
    prob = rf.predict_proba(input_sc)[0]
    
    injury_names = ['No Injury', 'Minor Injury', 'Serious Injury', 'Fatal']
    risk_levels = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']
    
    print("="*50)
    print("PREDICTION RESULT")
    print("="*50)
    print(f"Risk Level: {risk_levels[pred]}")
    print(f"Predicted Outcome: {injury_names[pred]}")
    print("\nProbabilities:")
    for name, p in zip(injury_names, prob):
        print(f"  {name}: {p*100:.1f}%")

# Example: 2-car collision, weekday, afternoon, 2pm, medium risk, not rush hour
predict_risk(
    collision=2,      # 2-Car
    weekend=0,        # Weekday
    time_period=1,    # Afternoon
    hour=14,          # 2 PM
    risk_factor=1,    # Medium Risk
    rush_hour=0       # No
)

---

## 🚀 Run Streamlit Dashboard

To run the full interactive Streamlit dashboard, open a terminal and run:

```bash
cd car_risk_system
streamlit run app.py
```